# 06 — Station Value Extraction

Extracts aligned raster values at gauge station coordinates.

In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError(
        "Project root was not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for folder in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Project root: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh


In [2]:
import pandas as pd
import rasterio
from pyproj import Transformer

gauge_path = PROCESSED_DIR / "station_samples" / "gauge_monthly_clean.csv"
gauge = pd.read_csv(gauge_path)

required = {"latitude", "longitude"}
if not required.issubset(gauge.columns):
    raise ValueError("Gauge CSV must contain latitude and longitude columns.")

aligned_files = sorted((PROCESSED_DIR / "aligned_rasters").rglob("*.tif"))
if not aligned_files:
    raise FileNotFoundError("Aligned rasters are missing. Run Notebook 05 first.")

In [3]:
samples = gauge.copy()

for raster_path in aligned_files:
    column_name = f"{raster_path.parent.name}_{raster_path.stem}"

    with rasterio.open(raster_path) as src:
        transformer = Transformer.from_crs(
            "EPSG:4326", src.crs, always_xy=True
        )
        xy = [
            transformer.transform(lon, lat)
            for lon, lat in zip(samples["longitude"], samples["latitude"])
        ]
        values = [value[0] for value in src.sample(xy)]
        samples[column_name] = values

output_path = PROCESSED_DIR / "station_samples" / "station_raster_samples.csv"
samples.to_csv(output_path, index=False)
display(samples.head())
print(f"Saved: {output_path}")

,station_id,station,latitude,longitude,year,month,date,rainfall_mm,CCS_2017_01,CCS_2017_02,...,NDVI_NDVI_2022_12,NDVI_NDVI_2022_2,NDVI_NDVI_2022_3,NDVI_NDVI_2022_4,NDVI_NDVI_2022_5,NDVI_NDVI_2022_6,NDVI_NDVI_2022_7,NDVI_NDVI_2022_8,NDVI_NDVI_2022_9,Slope_Slope_Degree_30m
0,CL503,Chalna,22.6012,89.5195,2017,1,2017-01-01,0.0,0.401950,0.0,...,0.7118,0.2923,0.2378,0.4964,0.3007,0.3314,0.4318,0.5576,0.4725,1.720395
1,CL510,Khulna,22.8319,89.5500,2017,1,2017-01-01,0.0,1.000000,0.0,...,0.4292,0.4341,0.4510,0.4259,0.4387,0.4287,0.4287,0.5311,0.5311,2.105171
2,CL504,Dumuria,22.8093,89.4145,2017,1,2017-01-01,0.0,1.000000,0.0,...,0.5399,0.6193,0.6278,0.5359,0.4492,0.6402,0.6551,0.7131,0.7113,1.932987
3,CL509,Kapilmuni,22.6887,89.3088,2017,1,2017-01-01,0.0,-9999.000000,-9999.0,...,0.4581,0.4407,0.5131,0.4759,0.4038,0.5365,0.5430,0.5698,0.6299,NaN
4,CL515,Paikgacha,22.5850,89.3182,2017,1,2017-01-01,0.0,0.625072,0.0,...,0.3852,0.4355,0.3429,0.3011,0.3580,0.2165,0.4173,0.4565,0.4611,1.600269


Saved: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\processed\station_samples\station_raster_samples.csv
